# Transformer — Full Working Explained

This notebook contains a structured explanation of how the Transformer works, from tokenization to attention, encoder/decoder blocks, and training.

## 1. Main Idea

The Transformer processes all tokens in parallel instead of reading them one by one like an RNN. Its core innovation is **attention**, which learns which words should focus on which other words.

## 2. High-Level Architecture

```text
Input Sentence
      ↓
Tokenization (text into smaller parts called token , tokens into numerical ids for model learning )
      ↓
Embedding (TokenIds becomes dense vector and stores sematic similarities )
      ↓
Positional Encoding ( letting the model know the position of the each word)
      ↓
Encoder Stack (undertand the input , makes the relationship )
      ↓
Decoder Stack ( produces the output , on the relationship )
      ↓
Linear Layer
      ↓
Logits (raw output produced by the Model)
      ↓
Temperature (divides the logit by temperature , it defines the predition of the model )
      ↓
Softmax 
      ↓
Predicted Token
```

## 3. Input Processing and TOKENIZATION

Example sentence:

`The cat sat on mat`

The sentence is split into smaller parts called tokens, then each token is converted into an integer ID called TokenIDs.

## 4. Embedding Layer

TokenIDs are converted into dense vectors.

Example:

`"The" → [0.2, -0.8, 1.1, ...]`

Embeddings capture semantic (“related to meaning”) similarity, so words with related meanings get nearby vectors.

TokenIds are arbitary , but the dense vectors are meaningful mathematical representation

## 5. Positional Encoding

The Transformer does not naturally know word order, so positional information is added to each embedding.

Without positional encoding, `Dog bites man` and `Man bites dog` could look too similar.

The original Transformer used sine/cosine positional encoding:

$$
PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/d_{model}}}\right),\quad
PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

## 6. Encoder

The encoder extracts meaning from the input sentence. In the original Transformer, several encoder blocks are stacked.

Each encoder block contains:

```text
Multi-Head Attention
        ↓
Add & Normalize
        ↓
Feed Forward Network
        ↓
Add & Normalize
```

### However, modern Transformers (e.g., GPT, Llama, BERT large variants) often use Pre-Layer Norm:
Normalize before the Multi-Head Attention or FFN , 	Then add the residual connection with the original input . ( if the layer are less and model is small < 20  then post normalization is better but if the layer are in the huge number > 20 then the pre normalization is the better option )

because :

	1. Better Gradient Flow
		With Post-Norm: Gradients can shrink or become unstable.
		With Pre-Norm: There is a clean path through residual connections. This makes optimization much easier.
		
	2. Deep Models Train More Reliably
		Original transformers worked for:
			6 layers
			12 layers

		Modern LLMs use:
			48 layers
			80 layers
			120+ layers

		Post-Norm often becomes unstable at these depths. Pre-Norm remains stable.

	3. Less gradient Vanishing 
		makes the gradient get preserved and remain a good amount so that model could learn 

	4. Faster Convergence
		Pre-Norm models usually:
			require less tuning
			train faster
			are more stable
		during large-scale training.

## 7. Self-Attention

Self-attention is the most important part of the Transformer.
Every token looks at every other token.  

Example sentence:

`The animal didn't cross the street because it was tired.`

Attention helps the model understand that `it` likely refers to `animal`.

## 8. Query, Key, Value

Each token creates three vectors:

- **Query (Q)**: what am I looking for?
- **Key (K)**: what information do I contain?
- **Value (V)**: what information should be passed on?

These vectors are used to compute attention scores.

## 9. Attention Formula

The standard attention equation is:

$$
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Steps:
1. Compute similarity using \(QK^T\)
2. Scale by \(\sqrt{d_k}\)
3. Apply softmax to get probabilities
4. Multiply by \(V\) to get the final context-aware output

## 10. Multi-Head Attention

Instead of using one attention mechanism, the Transformer uses multiple attention heads in parallel.

Different heads can focus on different patterns, such as grammar, pronouns, tense, or long-distance relationships.

## 11. Feed Forward Network

After attention, each token independently passes through a small neural network:

```text
Linear
 ↓
Activation (ReLU/GELU)
 ↓
Linear
```

This helps the model build richer features for each token.

## 12. Residual Connections and Layer Normalization

Residual connections help gradients flow during training:

$$
\text{Output} = \text{Layer}(x) + x
$$

Layer Normalization in the Transformer is a normalization technique that stabilizes and accelerates training by normalizing the activations across the feature dimension  (i.e., across the hidden units for each individual sample). Layer normalization stabilizes activations and makes training easier.

Why Do We Need layer normalization ?

    Suppose a layer produces these values: [1000, 2000, 3000]

    and another produces: [0.001, 0.002, 0.003]

    Such huge differences make training unstable:
        Gradients may explode.
        Gradients may vanish.
        Training becomes slower.
        The model may fail to converge.

    LayerNorm rescales the values to a more manageable range.

    layer normaization makes them on same level so weight updates constantly

## 13. Decoder

The decoder generates output tokens one by one.

Original Transformer decoder blocks contain:

```text
Masked Self Attention
↓
Encoder-Decoder Attention
↓
Feed Forward
```

## 14. Masked Attention

Masked attention prevents the decoder from seeing future tokens while predicting the next token.

This is essential for generation tasks like translation, completion, and chat.

## 15. Encoder-Decoder Attention

In translation tasks, the decoder attends to the encoder outputs to decide which input parts matter most while generating each output token.

## 16. Final Prediction and SOFTMAX 

The decoder output is passed through a linear layer and then a softmax function.

Softmax turns raw scores into probabilities, and the highest-probability token is selected.

## 17. Training

Training uses:

- **Loss function**: usually cross-entropy
- **Optimization**: often Adam
- **Backpropagation**: adjusts weights to reduce prediction error

## 18. Why Transformers Became Revolutionary

Compared to RNNs and LSTMs, Transformers are:

- faster to train
- easier to parallelize
- better at long-range dependencies
- highly scalable

## 19. Common Transformer Types

### Encoder-only
Used for understanding tasks like classification and search.

### Decoder-only
Used for generation tasks like chat and code completion.

### Encoder-decoder
Used for translation and summarization.

## 20. Full Flow

```text
Sentence
↓
Tokenization
↓
Embedding
↓
Positional Encoding
↓
Self Attention
↓
Feed Forward
↓
Encoder Stack
↓
Decoder Stack
↓
Linear
↓
Softmax
↓
Predicted Token
```

## 21. Simple Real-World Analogy

Think of a classroom discussion. Each student listens to every other student, decides who is important, and gathers the most relevant information. That is the idea behind attention.

## 22. One-Line Summary

The Transformer converts words into vectors, uses attention to relate every token to every other token, and repeats this process through stacked blocks to understand or generate language.

In [ ]:
# Example: conceptual attention shapes
seq_len = 5
d_model = 512
d_k = 64
print('Sequence length:', seq_len)
print('Model dimension:', d_model)
print('Key/query dimension:', d_k)


## Temperature

Temperature controls how random or confident a model’s predictions become during generation.

Low Temperature

    The highest logit becomes much more dominant.

    Model becomes: confident , deterministic , repetitive

High Temperature 

    Probabilities become more spread out.

    Model becomes: creative , random , exploratory

# LayerNorm vs BatchNorm
    LayerNorm	                    BatchNorm
    Normalizes across features	    Normalizes across batch samples
    Works well for NLP                  Originally designed for CNNs
    Independent of batch size	    Depends on batch statistics
    Used in Transformers	            Used heavily in vision models

# Why Transformers Prefer LayerNorm

    Transformers process sequences of varying lengths.

    BatchNorm becomes inconvenient because:
        Batch sizes may vary.
        Sequence lengths vary.
        Distributed training becomes harder.

    LayerNorm avoids these issues because each token is normalized independently.